# Module 5 - Compound Agent (Genie + KA + fetch + parse_pdf) (fixed)

In [ ]:
%run ./_config


In [ ]:
%pip install -q -U databricks-agents mlflow databricks-sdk
dbutils.library.restartPython()


In [ ]:
%run ./_config


In [ ]:
# Read upstream IDs from _workshop_config (populated by notebooks 03 + 04).

dbutils.widgets.text("genie_space_id", cfg_get('genie_space_id'), "Genie space id")
dbutils.widgets.text("ka_endpoint", cfg_get('ka_endpoint', f'agents_{KA_NAME}'), "KA serving endpoint")
dbutils.widgets.text("llm_endpoint", cfg_get('llm_endpoint', LLM_ENDPOINT), "Foundation model endpoint")

GENIE_SPACE_ID = dbutils.widgets.get("genie_space_id")
KA_ENDPOINT = dbutils.widgets.get("ka_endpoint")
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint")
assert GENIE_SPACE_ID, "genie_space_id is empty -- did you run notebook 03?"
print(f"GENIE={GENIE_SPACE_ID!r}  KA={KA_ENDPOINT!r}  LLM={LLM_ENDPOINT!r}")


## 1. Define the agent (writes agent.py to driver-local dir)

In [ ]:
import os
agent_dir = "/tmp/disa_agent"
os.makedirs(agent_dir, exist_ok=True)
agent_path = f"{agent_dir}/agent.py"

agent_src = '''"""DISA CTI compound agent."""
from __future__ import annotations

import json
import os
import re
from typing import Any

import mlflow
import requests
from databricks.sdk import WorkspaceClient
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse

GENIE_SPACE_ID = os.getenv("GENIE_SPACE_ID", "")
KA_ENDPOINT = os.getenv("KA_ENDPOINT", "agents_disa-cti-knowledge")
LLM_ENDPOINT = os.getenv("LLM_ENDPOINT", "databricks-claude-sonnet-4-6")

SYSTEM_PROMPT = """You are the DISA Cyber Threat Intelligence assistant.

You have five tools:
  - genie_query(question): structured questions over CVE / KEV / STIG / asset tables.
  - knowledge_assistant_search(question): retrieves passages from CISA advisories, DoD STIGs, ATT&CK.
  - fetch_advisory_url(url): fetches a public CISA / NVD / MITRE URL.
  - parse_pdf(volume_path): parses a PDF in a UC volume.
  - netops_outages_query(provider, days): real-world recent outage reports from the public outages mailing list.

Cite CVE IDs, STIG IDs, and ATT&CK technique IDs in your answers.
"""

TOOLS = [
    {"type": "function", "function": {"name": "genie_query", "description": "Ask a natural-language question over CVE/KEV/STIG/asset tables.", "parameters": {"type": "object", "properties": {"question": {"type": "string"}}, "required": ["question"]}}},
    {"type": "function", "function": {"name": "knowledge_assistant_search", "description": "Retrieve passages from CISA advisories, DoD STIGs, ATT&CK.", "parameters": {"type": "object", "properties": {"question": {"type": "string"}}, "required": ["question"]}}},
    {"type": "function", "function": {"name": "fetch_advisory_url", "description": "Fetch a public web URL and return text.", "parameters": {"type": "object", "properties": {"url": {"type": "string"}}, "required": ["url"]}}},
    {"type": "function", "function": {"name": "parse_pdf", "description": "Parse a PDF in a UC volume via ai_parse_document.", "parameters": {"type": "object", "properties": {"volume_path": {"type": "string"}}, "required": ["volume_path"]}}},
    {"type": "function", "function": {"name": "netops_outages_query", "description": "Look up real network operator outage reports (from the public outages mailing list, ai_query-classified). Useful for questions about recent BGP/DNS/peering/fiber/etc incidents affecting carriers and ISPs.", "parameters": {"type": "object", "properties": {"provider": {"type": "string", "description": "Carrier/ISP/AS name to filter by, or empty string for all."}, "days": {"type": "integer", "description": "Lookback window in days. Defaults to 30."}}, "required": ["provider"]}}},
]


class CTIAgent(ResponsesAgent):
    def __init__(self) -> None:
        self.w = WorkspaceClient()

    def _serving_invoke(self, endpoint: str, body: dict) -> dict:
        # Use SDK api_client.do() which handles all auth modes (PAT, OAuth, M2M, on-behalf-of).
        return self.w.api_client.do("POST", f"/serving-endpoints/{endpoint}/invocations", body=body)

    def _llm_call(self, messages, tools=None):
        body = {"messages": messages}
        if tools is not None:
            body["tools"] = tools
        return self._serving_invoke(LLM_ENDPOINT, body)

    def _genie_query(self, question: str) -> str:
        if not GENIE_SPACE_ID:
            return "Genie space not configured."
        try:
            conv = self.w.genie.start_conversation_and_wait(space_id=GENIE_SPACE_ID, content=question)
            attachments = conv.attachments or []
            if not attachments:
                return conv.content or "No response."
            result = self.w.genie.get_message_attachment_query_result(
                space_id=GENIE_SPACE_ID,
                conversation_id=conv.conversation_id,
                message_id=conv.message_id,
                attachment_id=attachments[0].attachment_id,
            )
            return json.dumps(result.statement_response.result.data_array[:20])
        except Exception as e:
            return f"genie_query failed: {e}"

    def _ka_search(self, question: str) -> str:
        try:
            resp = self._serving_invoke(KA_ENDPOINT, {"messages": [{"role": "user", "content": question}]})
            return resp["choices"][0]["message"]["content"]
        except Exception as e:
            return f"knowledge_assistant_search failed (KA endpoint may not be deployed): {e}"

    def _fetch_url(self, url: str) -> str:
        import requests
        if not url.startswith(("https://www.cisa.gov", "https://nvd.nist.gov", "https://attack.mitre.org")):
            return f"Refusing to fetch {url}: only CISA / NVD / MITRE domains allowed."
        try:
            r = requests.get(url, timeout=15, headers={"User-Agent": "DISA-Workshop"})
            text = re.sub(r"<[^>]+>", " ", r.text)
            text = re.sub(r"\\s+", " ", text).strip()
            return text[:8000]
        except Exception as e:
            return f"Fetch failed: {e}"

    def _parse_pdf(self, volume_path: str) -> str:
        if not volume_path.startswith("/Volumes/"):
            return "Refusing to parse: path must be under /Volumes/."
        try:
            from pyspark.sql import SparkSession
            spark = SparkSession.builder.getOrCreate()
            df = spark.sql(
                f"SELECT LEFT(CAST(ai_parse_document(content) AS STRING), 6000) AS preview "
                f"FROM READ_FILES(\\'{volume_path}\\', format => \\'binaryFile\\')"
            )
            row = df.collect()[0]
            return row.preview or ""
        except Exception as e:
            return f"Parse failed: {e}"


    def _netops_outages_query(self, provider: str, days: int) -> str:
        try:
            from pyspark.sql import SparkSession
            spark = SparkSession.builder.getOrCreate()
            base = os.getenv("WORKSHOP_BASE", "")
            if not base:
                return "WORKSHOP_BASE env var not set; cannot route to per-user netops_outages table."
            df = spark.sql(
                f"SELECT sent_at, subject, root_cause, severity, status, summary, affected_providers FROM {base}.netops_outages "
                f"WHERE is_outage_report = true AND sent_at >= date_sub(current_timestamp(), {days}) "
                + (f"AND exists(affected_providers, p -> lower(p) LIKE \"%{provider.lower()}%\") " if provider else "")
                + "ORDER BY sent_at DESC LIMIT 20"
            )
            return json.dumps([row.asDict() for row in df.collect()], default=str)
        except Exception as e:
            return f"netops_outages_query failed: {e}"

    def _dispatch(self, name: str, args: dict) -> str:
        if name == "genie_query":
            return self._genie_query(args["question"])
        if name == "knowledge_assistant_search":
            return self._ka_search(args["question"])
        if name == "fetch_advisory_url":
            return self._fetch_url(args["url"])
        if name == "parse_pdf":
            return self._parse_pdf(args["volume_path"])
        if name == "netops_outages_query":
            return self._netops_outages_query(args.get("provider", ""), int(args.get("days", 30)))
        return f"Unknown tool {name}"

    def predict(self, request) -> ResponsesAgentResponse:
        if hasattr(request, "input"):
            msgs_in = request.input
        else:
            msgs_in = request.get("input", []) if isinstance(request, dict) else []
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        def _flatten(content):
            if isinstance(content, str):
                return content
            if content is None:
                return ""
            if isinstance(content, list):
                parts = []
                for p in content:
                    t = getattr(p, "text", None)
                    if t is None and isinstance(p, dict):
                        t = p.get("text") or p.get("content")
                    if t:
                        parts.append(str(t))
                return "\\n".join(parts)
            return str(content)

        for m in msgs_in:
            role = getattr(m, "role", None)
            if role is None and isinstance(m, dict):
                role = m.get("role")
            raw = getattr(m, "content", None)
            if raw is None and isinstance(m, dict):
                raw = m.get("content")
            messages.append({"role": role, "content": _flatten(raw)})

        import uuid as _uuid
        def _make_response(text: str) -> ResponsesAgentResponse:
            return ResponsesAgentResponse(output=[{
                "id": f"msg_{_uuid.uuid4().hex}",
                "type": "message",
                "role": "assistant",
                "content": [{"type": "output_text", "text": text}],
            }])

        for _ in range(6):
            resp = self._llm_call(messages, tools=TOOLS)
            choice = resp["choices"][0]["message"]
            tool_calls = choice.get("tool_calls") or []
            if not tool_calls:
                return _make_response(choice.get("content") or "")
            messages.append({"role": "assistant", "content": choice.get("content") or "", "tool_calls": tool_calls})
            for tc in tool_calls:
                args = json.loads(tc["function"]["arguments"])
                result = self._dispatch(tc["function"]["name"], args)
                messages.append({"role": "tool", "tool_call_id": tc.get("id"), "content": str(result)[:6000]})

        return _make_response("Hit max tool-call iterations.")


mlflow.models.set_model(CTIAgent())
'''

with open(agent_path, "w") as f:
    f.write(agent_src)
print(f"Wrote {agent_path} ({len(agent_src)} bytes)")


## 2. Smoke test the agent locally (LLM-only, no Genie/KA dependencies)

In [ ]:
os.environ["GENIE_SPACE_ID"] = GENIE_SPACE_ID
os.environ["KA_ENDPOINT"] = KA_ENDPOINT
os.environ["LLM_ENDPOINT"] = LLM_ENDPOINT

import sys
sys.path.insert(0, agent_dir)
import importlib
if "agent" in sys.modules:
    importlib.reload(sys.modules["agent"])
from agent import CTIAgent  # noqa
agent = CTIAgent()

class _Req:
    def __init__(self, msgs):
        self.input = msgs

req = _Req([{"role": "user", "content": "List the 4 tools you have available, in one line each. Do not call any tool."}])
resp = agent.predict(req)
last = resp.output[-1]
# OutputItem is a pydantic model; access via .model_dump() for portability
last_d = last.model_dump() if hasattr(last, "model_dump") else last
content = last_d.get("content")
if isinstance(content, list) and content:
    text = content[0].get("text", "")
else:
    text = str(content or "")
print("AGENT OUTPUT:", text[:1500])

## 3. Log + register the agent to UC

In [ ]:
import mlflow
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksFunction

mlflow.set_registry_uri("databricks-uc")

resources = [
    DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT),
    DatabricksFunction(function_name=f"{BASE}.lookup_cve"),
    DatabricksFunction(function_name=f"{BASE}.assets_for_product"),
]
# Only include the KA endpoint resource if the user has actually configured one
if KA_ENDPOINT and KA_ENDPOINT != "agents_disa-cti-knowledge":
    resources.append(DatabricksServingEndpoint(endpoint_name=KA_ENDPOINT))

with mlflow.start_run(run_name="disa-cti-agent"):
    info = mlflow.pyfunc.log_model(
        artifact_path="agent",
        python_model=agent_path,
        registered_model_name=AGENT_MODEL,
        pip_requirements=["mlflow", "databricks-sdk", "databricks-agents", "requests", "pyspark"],
        resources=resources,
    )
print(f"Logged model: {info.model_uri}")
print(f"Registered version: {info.registered_model_version}")


## 4. Deploy to Model Serving as `disa-cti-agent`

In [ ]:
from databricks.agents import deploy

env_vars = {
    "GENIE_SPACE_ID": GENIE_SPACE_ID,
    "KA_ENDPOINT": KA_ENDPOINT,
    "LLM_ENDPOINT": LLM_ENDPOINT,
    "WORKSHOP_BASE": BASE,
}

deployment = deploy(
    model_name=AGENT_MODEL,
    model_version=info.registered_model_version,
    endpoint_name=AGENT_ENDPOINT,
    environment_vars=env_vars,
)
print(f"Deployment: {deployment}")
